# Governance Helper (small demo) — classify & score

**Lloyds · AI Agent Security & Governance · Module 8 Day 1 · Dr. Deepali Salwan**

A *tiny* hands-on to make two governance ideas concrete before the case study:

1. **Risk classification** — walk any AI system through the EU AI Act decision tree in code.
2. **Risk register scoring** — turn a list of risks into a scored, sorted, flagged register.

> **Offline. No API key. Standard library (+ one small chart).** This is a thinking aid, not a
> compliance tool — governance is a human decision; this just shows the mechanics.


## 1 · Classify an AI system (EU AI Act tiers)

The decision tree: **prohibited?** → **Annex III (high-risk)?** → **interacts with people
(limited)?** → else **minimal**. The first "yes" wins. We classify by the *most capable* thing the
agent can do, and govern UP when unsure.


In [ ]:
def classify(name, prohibited=False, annex_iii=False, interacts=False):
    if prohibited:  tier = "PROHIBITED (banned)"
    elif annex_iii: tier = "HIGH-RISK"
    elif interacts: tier = "LIMITED (transparency duties)"
    else:           tier = "MINIMAL"
    return name, tier

# three systems, classified by their MOST capable action
systems = [
    classify("PennyPilot (payments: own-account balance + transfers)",
             annex_iii=False, interacts=True),
    classify("LendLite (assesses personal-loan affordability & suggests a decision)",
             annex_iii=True,  interacts=True),   # creditworthiness -> Annex III 5(b)
    classify("FraudSpotter (flags suspicious transactions for review)",
             annex_iii=False, interacts=False),  # fraud detection is carved OUT of 5(b)
]
print(f"{'system':62s} tier")
print("-"*90)
for name, tier in systems:
    print(f"{name:62s} {tier}")
print("\nNote: LendLite evaluates creditworthiness -> Annex III 5(b) -> HIGH-RISK.")
print("FraudSpotter also judges customers, but fraud detection is explicitly EXCLUDED from 5(b).")


## 2 · Score a risk register

Each risk gets a **likelihood** and an **impact** (High=3 / Med=2 / Low=1). Score = L x I. We sort
highest-first and flag anything above the **acceptance threshold** — those block go-live until
mitigated or consciously accepted by a named owner.


In [ ]:
LVL = {"H": 3, "M": 2, "L": 1}
def level(score):
    return "HIGH" if score >= 6 else "MEDIUM" if score >= 3 else "LOW"

# a starter register for LendLite (the case-study system)
register = [
    # risk, likelihood, impact, control, owner
    ("Biased loan decisions across protected groups", "M", "H",
     "Bias testing on the training data + human review of declines", "Head of AI Risk"),
    ("Wrong affordability decision harms a customer", "M", "H",
     "Human-in-the-loop sign-off on every decision (Art 14)", "Lending Lead"),
    ("Applicant PII exposed / mis-used", "L", "H",
     "Encryption, access control, UK GDPR lawful basis (M5)", "Security Lead"),
    ("No explanation for a decision (opacity)", "H", "M",
     "Reason codes + model/agent card (Art 13)", "Product"),
    ("Prompt injection via application free-text", "M", "M",
     "Treat application text as data; red-team gate (M7)", "Security Lead"),
]

ACCEPT = 4   # anything scoring above this must be fixed or formally accepted

scored = []
for risk, l, i, control, owner in register:
    s = LVL[l] * LVL[i]
    scored.append((s, level(s), risk, f"{l}x{i}", control, owner))
scored.sort(reverse=True)

print(f"{'score':>5} {'level':7} {'L x I':6} risk")
print("-"*78)
for s, lvl, risk, li, control, owner in scored:
    flag = "  <-- BLOCKS GO-LIVE" if s > ACCEPT else ""
    print(f"{s:>5} {lvl:7} {li:6} {risk}{flag}")
blockers = sum(1 for s,*_ in scored if s > ACCEPT)
print(f"\n{blockers} risk(s) above the acceptance threshold ({ACCEPT}) -> must be fixed or owned before launch.")


In [ ]:
# a small picture: how many risks sit at each level
import matplotlib.pyplot as plt
from collections import Counter
counts = Counter(lvl for _, lvl, *_ in scored)
order = ["HIGH", "MEDIUM", "LOW"]
vals = [counts.get(k, 0) for k in order]
fig, ax = plt.subplots(figsize=(5.5, 3))
ax.bar(order, vals, color=["#B01818", "#C8860A", "#2C7A4B"])
for x, v in enumerate(vals):
    ax.text(x, v + 0.03, str(v), ha="center", fontweight="bold")
ax.set_ylabel("risks"); ax.set_title("LendLite risk register by level")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()


## What this showed (and what it did not)

- **Classification is a decision, not a lookup** — LendLite is high-risk because of what it
  *decides* (creditworthiness), while PennyPilot (payments) and FraudSpotter (a carve-out) are not.
- **A risk register is just L x I, sorted, with a threshold** — but the *judgement* is in scoring
  honestly and naming an owner for every risk you accept.

What a tiny script can't do: decide whether a residual risk is acceptable, or who is accountable.
That is the human governance work — which is exactly the case study you do next.

> **Try it:** change a likelihood or impact and watch the go-live blockers change; add a row for a
> risk the register is missing; lower `ACCEPT` to 2 and see how much stricter the gate becomes.
